In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
import os
import pynbody
import pynbody.plot.sph as sph
from IPython.display import clear_output, display

# fourier
from scipy.signal import stft as short_time_fft
from scipy.ndimage import gaussian_filter1d
from scipy.signal import ShortTimeFFT, find_peaks, get_window
from scipy.signal import windows

# dimensionality reduction
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

SMALLSIZE = 12
NORMALSIZE = 12
LARGESIZE = 12

params = {
    "font.family":"serif",
    "mathtext.fontset":"stix",
    "font.size": SMALLSIZE,
    "axes.titlesize" : SMALLSIZE,
    "axes.labelsize" : NORMALSIZE,
    'xtick.labelsize': SMALLSIZE,
    'ytick.labelsize': SMALLSIZE,      
    'legend.fontsize': SMALLSIZE,  
    'figure.titlesize': LARGESIZE,
    'pgf.texsystem' : "pdflatex"
}
matplotlib.rcParams.update(params)

isob_dir = "../data/IsoB_dt10_all"
snap_names = [f"GLX.000{10*(j):03.0f}" for j in range(1, 62)] + [f"GLX.0{10*j:04.0f}" for j in range(62, 101)]


In [ ]:
#print(star_frame)
#tafasd = pd.DataFrame(temp_stack[0, :, :].transpose(), columns=["iord", "x", "y", "z", "rz", "phiz", "mass", "snap"])

#temp = pd.DataFrame(star_frame[0, :, :, 0].transpose(), columns=["iord", "x", "y", "z", "rz", "phiz", "mass", "snap"])

dfs = []
iords = []
morphs = []
s1000r = []

for morph in range(len(star_frame[0, 0, 0, :])): # this is the amount of morphologies
    sf_by_morph = star_frame[:, :, :, morph]
    for star_num in range(len(sf_by_morph[:, 0, 0])):
        sdf = pd.DataFrame(sf_by_morph[star_num, :, :].transpose(), columns=["iord", "x", "y", "z", "rz", "phiz", "mass", "snap"])
        dfs.append(sdf)
        iords.append(sdf['iord'].to_numpy()[0])
        morphs.append(morph)
        s1000r.append(np.sqrt(sdf['x'].to_numpy()[-1]**2 + sdf['y'].to_numpy()[-1]**2))


In [ ]:
stars_df = pd.DataFrame(dict(
    iord = iords,
    morph = morphs,
    s1000r = s1000r,
    df = dfs
))
print(stars_df)

In [ ]:
sigma_val = 1

dataframes = []

for i, sdf in enumerate(stars_df['df']):
    print(i)
    x = sdf['x'].dropna().to_numpy()
    y = sdf['y'].dropna().to_numpy()
    z = sdf['z'].dropna().to_numpy()
    t = sdf['snap'].dropna().to_numpy()
    #m = sdf['mass'].dropna().to_numpy()
    #snum = stars_df['star_num'].to_numpy()[i]


    dt, N = 10, len(t)
    fs = 1 / dt

    win_size = 20 # with a window size of 10 samples, one window spans 200 Myr

    win = windows.hann(win_size)
    hop = int(win_size * 0.5) # 50% overlap

    
    f, t, X_stft = short_time_fft(x, fs=fs, nperseg=win_size)
    f, t, Y_stft = short_time_fft(y, fs=fs, nperseg=win_size)
    f, t, Z_stft = short_time_fft(z, fs=fs, nperseg=win_size)
    # plt.clf()
    # plt.pcolormesh(t, f, np.abs(Zxx), shading='nearest')
    # plt.ylabel("[Myr^-1]")
    # plt.xlabel("Myr")
    # plt.annotate(snum, xy=(0.95, 0.95), xycoords='axes fraction', va='top', ha='right', color="white", fontsize=20)
    # plt.show()

    # print(np.abs(Z_stft)[:, 0].shape) # this extracts a vertical slice at t_index=0, of size 11 which matches the frequencies
    # plt.clf()
    # fig, ax = plt.subplots()

    X_fwmean = np.zeros(len(t), dtype=np.float64) # frequency weighted mean
    Y_fwmean = np.zeros(len(t), dtype=np.float64)
    Z_fwmean = np.zeros(len(t), dtype=np.float64)

    X_fwstd = np.zeros(len(t), dtype=np.float64) # frequency weighted standard deviation
    Y_fwstd = np.zeros(len(t), dtype=np.float64)
    Z_fwstd = np.zeros(len(t), dtype=np.float64)

    X_amp = np.zeros(len(t), dtype=np.float64)
    Y_amp = np.zeros(len(t), dtype=np.float64)
    Z_amp = np.zeros(len(t), dtype=np.float64)

    X_amp_std = np.zeros(len(t), dtype=np.float64)
    Y_amp_std = np.zeros(len(t), dtype=np.float64)
    Z_amp_std = np.zeros(len(t), dtype=np.float64)

    for time_index, time_slice in enumerate(t):
        x_vals = np.abs(X_stft)[:, time_index]
        y_vals = np.abs(Y_stft)[:, time_index]
        z_vals = np.abs(Z_stft)[:, time_index]

        # ax.plot(f, x_vals)

        # ax.vlines(np.divide(np.sum(np.multiply(x_vals, f)), np.sum(x_vals)), ymin=x_vals.min(), ymax=x_vals.max())
        # ax.hlines(x_vals.max(), xmin=0, xmax=0.05)


        # x_gauss = gaussian_filter1d(x_vals, sigma=sigma_val)
        # y_gauss = gaussian_filter1d(y_vals, sigma=sigma_val)
        # z_gauss = gaussian_filter1d(z_vals, sigma=sigma_val)

        # xpeaks, xprop = find_peaks(x_vals)
        # ypeaks, yprop = find_peaks(y_vals)
        # zpeaks, zprop = find_peaks(z_vals,)

        #print(len(xpeaks), len(ypeaks), len(zpeaks))

        #ax.plot(f, z_vals, color='green')

        #im = ax.plot(f, z_vals)

        #color = im[0].get_color()
        

        X_fwmean[time_index] = np.divide(np.sum(np.multiply(x_vals, f)), np.sum(x_vals))
        Y_fwmean[time_index] = np.divide(np.sum(np.multiply(y_vals, f)), np.sum(y_vals))
        Z_fwmean[time_index] = np.divide(np.sum(np.multiply(z_vals, f)), np.sum(z_vals))

        X_fwstd[time_index] = np.sqrt(np.sum(x_vals * (f - X_fwmean[time_index]) ** 2)/np.sum(x_vals))
        Y_fwstd[time_index] = np.sqrt(np.sum(y_vals * (f - Y_fwmean[time_index]) ** 2)/np.sum(y_vals))
        Z_fwstd[time_index] = np.sqrt(np.sum(z_vals * (f - Z_fwmean[time_index]) ** 2)/np.sum(z_vals))

        X_amp[time_index] = x_vals.mean()
        Y_amp[time_index] = y_vals.mean()
        Z_amp[time_index] = z_vals.mean()

        X_amp_std[time_index] = x_vals.std()
        Y_amp_std[time_index] = y_vals.std()
        Z_amp_std[time_index] = z_vals.std()





        #print(time_index, avg_fx)

        # ax.scatter(time_slice, avg_fz)
        #ax.vlines(avg_fz, ymin=z_vals.min(), ymax=z_vals.max(), colors=color)

    processed_df = pd.DataFrame(dict(
        x_freq_wmean = X_fwmean,
        y_freq_wmean = Y_fwmean,
        z_freq_wmean = Z_fwmean,
        x_freq_wstd = X_fwstd,
        y_freq_wstd = Y_fwstd,
        z_freq_wstd = Z_fwstd,
        x_amp_mean = X_amp,
        y_amp_mean = Y_amp,
        z_amp_mean = Z_amp,
        x_amp_std = X_amp_std,
        y_amp_std = Y_amp_std,
        z_amp_std = Z_amp_std,
    ))
    
    print(processed_df)
    dataframes.append(processed_df)
    # ax.errorbar(t, Z_fwmean, yerr=X_fwstd)

    # axx = ax.twinx()

    # axx.plot(t, Z_amp, color='red')

    # # ax.plot(t, X_fwmean)
    # # ax.plot(t, Y_fwmean, c="red")
    # # ax.plot(t, Z_fwmean, c="green")
    # ax.set(ylim=(f.min(), f.max()))
    # # print(X_fwmean)
        
    # plt.show()





    # there are 11 frequency



In [ ]:
morph_colours = ["blue" if morphs[j] == 1 else "purple" if morphs[j] == 0 else "green" for j in range(len(stars_df))]
radial_colours = stars_df['s1000r']

In [ ]:
datacube = np.array(dataframes)

val1 = np.array([datacube[j, :, 0].mean() for j in range(len(stars_df))]) # xyz freq wmean
val2 = np.array([datacube[j, :, 1].mean() for j in range(len(stars_df))])
val3 = np.array([datacube[j, :, 2].mean() for j in range(len(stars_df))])
val4 = np.array([datacube[j, :, 3].mean() for j in range(len(stars_df))]) # xyz freq std wmean
val5 = np.array([datacube[j, :, 4].mean() for j in range(len(stars_df))])
val6 = np.array([datacube[j, :, 5].mean() for j in range(len(stars_df))])
val7 = np.array([datacube[j, :, 6].mean() for j in range(len(stars_df))]) # xyz amp wmean
val8 = np.array([datacube[j, :, 7].mean() for j in range(len(stars_df))])
val9 = np.array([datacube[j, :, 8].mean() for j in range(len(stars_df))])



fig, ax = plt.subplots()
#ax.scatter(val9, val10, s=3)
im = ax.scatter(val1, val4, s=5, c=radial_colours)
fig.colorbar(im, ax=ax, label="GLX.01000 Radius [kpc]")
ax.set(ylabel="$\\sigma$ Frequency of $x$ [Myr$^{-1}$]", xlabel="Frequency of $x$ [Myr$^{-1}$]")
plt.show()

fig, ax = plt.subplots()
#ax.scatter(val9, val10, s=3)
im = ax.scatter(val3, val6, s=5, c=radial_colours)
fig.colorbar(im, ax=ax, label="GLX.01000 Radius [kpc]")
ax.set(ylabel="$\\sigma$ Frequency of $z$ [Myr$^{-1}$]", xlabel="Frequency of $z$ [Myr$^{-1}$]")
plt.show()

fig, ax = plt.subplots()
#ax.scatter(val9, val10, s=3)
im = ax.scatter(val1, val7, s=5, c=radial_colours)
fig.colorbar(im, ax=ax, label="GLX.01000 Radius [kpc]")
ax.set(ylabel="Amplitude of oscillations in $x$ [kpc]", xlabel="Frequency of $x$ [Myr$^{-1}$]")
plt.show()

fig, ax = plt.subplots()
#ax.scatter(val9, val10, s=3)
im = ax.scatter(val3, val9, s=5, c=radial_colours)
fig.colorbar(im, ax=ax, label="GLX.01000 Radius [kpc]")
ax.set(ylabel="Amplitude of oscillations in $z$ [kpc]", xlabel="Frequency of $z$ [Myr$^{-1}$]")
plt.show()


fig, ax = plt.subplots()
#ax.scatter(val9, val10, s=3)
im = ax.scatter(val6, val9, s=5, c=radial_colours)
fig.colorbar(im, ax=ax, label="GLX.01000 Radius [kpc]")
ax.set(ylabel="Amplitude of oscillations in $z$ [kpc]", xlabel="Frequency of $z$ [Myr$^{-1}$]")
plt.show()

fig, ax = plt.subplots()
#ax.scatter(val9, val10, s=3)
im = ax.scatter(val4, val6, s=5, c=radial_colours)
fig.colorbar(im, ax=ax, label="GLX.01000 Radius [kpc]")
ax.set(ylabel="Amplitude of oscillations in $z$ [kpc]", xlabel="Frequency of $z$ [Myr$^{-1}$]")
plt.show()


# Dimensionality Reduction + Clustering

In [ ]:
val1 = np.array([datacube[j, :, 0].mean() for j in range(len(stars_df))])
val2 = np.array([datacube[j, :, 1].mean() for j in range(len(stars_df))])
val3 = np.array([datacube[j, :, 2].mean() for j in range(len(stars_df))])
val4 = np.array([datacube[j, :, 3].mean() for j in range(len(stars_df))])
val5 = np.array([datacube[j, :, 4].mean() for j in range(len(stars_df))])
val6 = np.array([datacube[j, :, 5].mean() for j in range(len(stars_df))])

kdf = pd.DataFrame(dict(
    x_freq_wmean = val1,
    y_freq_wmean = val2,
    z_freq_wmean = val3,
    x_freq_std = val4,
    y_freq_std = val5,
    z_freq_std = val6,
))

scaler = StandardScaler()

X_scaled = scaler.fit_transform(kdf)

kmeans = KMeans(n_clusters=10, random_state=1)
reg_clusters = kmeans.fit_predict(X_scaled)


fig, ax = plt.subplots()
ax.scatter(val5, val6, c=reg_clusters)

In [ ]:
val1 = np.array([datacube[j, :, 0].mean() for j in range(len(stars_df))])
val2 = np.array([datacube[j, :, 1].mean() for j in range(len(stars_df))])
val3 = np.array([datacube[j, :, 2].mean() for j in range(len(stars_df))])
val4 = np.array([datacube[j, :, 3].mean() for j in range(len(stars_df))])
val5 = np.array([datacube[j, :, 4].mean() for j in range(len(stars_df))])
val6 = np.array([datacube[j, :, 5].mean() for j in range(len(stars_df))])
val7 = np.array([datacube[j, :, 6].mean() for j in range(len(stars_df))])
val8 = np.array([datacube[j, :, 7].mean() for j in range(len(stars_df))])
val9 = np.array([datacube[j, :, 8].mean() for j in range(len(stars_df))])
val10 = np.array([datacube[j, :, 9].mean() for j in range(len(stars_df))])
val11 = np.array([datacube[j, :, 10].mean() for j in range(len(stars_df))])
val12 = np.array([datacube[j, :, 11].mean() for j in range(len(stars_df))])


pca_df = pd.DataFrame(dict(
    x_freq_wmean = val1,
    y_freq_wmean = val2,
    z_freq_wmean = val3,
    x_freq_std = val4,
    y_freq_std = val5,
    z_freq_std = val6,
    x_amp = val7,
    y_amp = val8,
    z_amp = val9,
    x_amp_std = val10,
    y_amp_std = val11,
    z_amp_std = val12,

))

pca_df2 = pd.DataFrame(dict(
    x_amp = val7,
    y_amp = val8,
    z_amp = val9,
))


scaler = StandardScaler()

X_scaled = scaler.fit_transform(pca_df)

pca = PCA()
X_pca = pca.fit(X_scaled)

eigen = X_pca.explained_variance_


plt.plot(range(1, len(eigen) + 1), eigen, marker='o')

plt.show()

In [ ]:
# k means elbow method

val1 = np.array([datacube[j, :, 0].mean() for j in range(len(stars_df))])
val2 = np.array([datacube[j, :, 1].mean() for j in range(len(stars_df))])
val3 = np.array([datacube[j, :, 2].mean() for j in range(len(stars_df))])
val4 = np.array([datacube[j, :, 3].mean() for j in range(len(stars_df))])
val5 = np.array([datacube[j, :, 4].mean() for j in range(len(stars_df))])
val6 = np.array([datacube[j, :, 5].mean() for j in range(len(stars_df))])
val7 = np.array([datacube[j, :, 6].mean() for j in range(len(stars_df))])
val8 = np.array([datacube[j, :, 7].mean() for j in range(len(stars_df))])
val9 = np.array([datacube[j, :, 8].mean() for j in range(len(stars_df))])
val10 = np.array([datacube[j, :, 9].mean() for j in range(len(stars_df))])
val11 = np.array([datacube[j, :, 10].mean() for j in range(len(stars_df))])
val12 = np.array([datacube[j, :, 11].mean() for j in range(len(stars_df))])


pca_df = pd.DataFrame(dict(
    x_freq_wmean = val1,
    y_freq_wmean = val2,
    z_freq_wmean = val3,
    x_freq_std = val4,
    y_freq_std = val5,
    z_freq_std = val6,
    x_amp = val7,
    y_amp = val8,
    z_amp = val9,
    x_amp_std = val10,
    y_amp_std = val11,
    z_amp_std = val12,

))
K = range(1,20)


    

scaler = StandardScaler()

X_scaled = scaler.fit_transform(pca_df)

pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_scaled)
inertias = []

for k in K:

    n_clusters = k
    kmeans = KMeans(n_clusters=n_clusters, random_state=123)
    pca_clusters = kmeans.fit(X_pca)
    inertias.append(kmeans.inertia_)

print(inertias)

plt.plot(K, inertias, 'bx-')

# ncols = 2
# nrows = 4
# fig, axes = plt.subplots(ncols=ncols, nrows=nrows)
# axes = axes.ravel()
# fig.set_size_inches(ncols*5, 4*nrows)
# # subplot_kw={"projection":"2d"}

# ax = axes[0]
# ax.scatter(X_pca[:, 0], X_pca[:, 1], c= X_pca[:, 2])
# ax = axes[1]
# ax.scatter(X_pca[:, 0], X_pca[:, 1], c= pca_clusters)
# ax = axes[2]
# ax.scatter(X_pca[:, 0], X_pca[:, 1], c= morph_colours, s=3, alpha=0.2)
# ax = axes[3]
# ax.scatter(X_pca[:, 0], X_pca[:, 1], c= radial_colours)
# ax = axes[4]
# cluster_mask = pca_clusters==0
# ax.scatter(X_pca[:, 0][cluster_mask], X_pca[:, 1][cluster_mask], c= pca_clusters[cluster_mask])
# ax.set(ylim=(-2, 1.5), xlim=(-3, 4))

# ax = axes[6]
# ax.scatter(X_pca[:, 0], X_pca[:, 2], c= pca_clusters)
# ax = axes[7]

# ax.scatter(X_pca[:, 1], X_pca[:, 2], c=pca_clusters)

In [ ]:
print(dataframes[0].columns)
val1 = np.array([datacube[j, :, 0].mean() for j in range(len(stars_df))])
val2 = np.array([datacube[j, :, 1].mean() for j in range(len(stars_df))])
val3 = np.array([datacube[j, :, 2].mean() for j in range(len(stars_df))])
val4 = np.array([datacube[j, :, 3].mean() for j in range(len(stars_df))])
val5 = np.array([datacube[j, :, 4].mean() for j in range(len(stars_df))])
val6 = np.array([datacube[j, :, 5].mean() for j in range(len(stars_df))])
val7 = np.array([datacube[j, :, 6].mean() for j in range(len(stars_df))])
val8 = np.array([datacube[j, :, 7].mean() for j in range(len(stars_df))])
val9 = np.array([datacube[j, :, 8].mean() for j in range(len(stars_df))])
val10 = np.array([datacube[j, :, 9].mean() for j in range(len(stars_df))])
val11 = np.array([datacube[j, :, 10].mean() for j in range(len(stars_df))])
val12 = np.array([datacube[j, :, 11].mean() for j in range(len(stars_df))])


pca_df = pd.DataFrame(dict(
    x_freq_wmean = val1,
    y_freq_wmean = val2,
    z_freq_wmean = val3,
    x_freq_std = val4,
    y_freq_std = val5,
    z_freq_std = val6,
    x_amp = val7,
    y_amp = val8,
    z_amp = val9,
    x_amp_std = val10,
    y_amp_std = val11,
    z_amp_std = val12,

))

pca_df2 = pd.DataFrame(dict(
    x_amp = val7,
    y_amp = val8,
    z_amp = val9,
))


scaler = StandardScaler()

X_scaled = scaler.fit_transform(pca_df)

pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_scaled)

n_clusters = 10
kmeans = KMeans(n_clusters=n_clusters, random_state=123)
pca_clusters = kmeans.fit_predict(X_pca)

ncols = 2
nrows = 4
fig, axes = plt.subplots(ncols=ncols, nrows=nrows)
axes = axes.ravel()
fig.set_size_inches(ncols*5, 4*nrows)
# subplot_kw={"projection":"2d"}

ax = axes[0]
ax.scatter(X_pca[:, 0], X_pca[:, 1], c= X_pca[:, 2])
ax = axes[1]
ax.scatter(X_pca[:, 0], X_pca[:, 1], c= pca_clusters)
ax = axes[2]
ax.scatter(X_pca[:, 0], X_pca[:, 1], c= morph_colours, s=3, alpha=0.2)
ax = axes[3]
ax.scatter(X_pca[:, 0], X_pca[:, 1], c= radial_colours)
ax = axes[4]
cluster_mask = pca_clusters==0
ax.scatter(X_pca[:, 0][cluster_mask], X_pca[:, 1][cluster_mask], c= pca_clusters[cluster_mask])
ax.set(ylim=(-2, 1.5), xlim=(-3, 4))

ax = axes[6]
ax.scatter(X_pca[:, 0], X_pca[:, 2], c= pca_clusters)
ax = axes[7]

ax.scatter(X_pca[:, 1], X_pca[:, 2], c=pca_clusters)

In [ ]:
tsne = TSNE(n_components=2, perplexity=50)
X_tsne = tsne.fit_transform(pca_df)

print(X_tsne.shape)

n_clusters = 7
kmeans = KMeans(n_clusters=n_clusters, random_state=15)
tsne_clusters = kmeans.fit_predict(X_tsne)

fig, axes = plt.subplots(ncols=2, nrows=2)
fig.set_size_inches(2*6, 2*4)
axes= axes.ravel()
ax=axes[0]
#ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=clusters)
im = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=tsne_clusters)
#im = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=X_tsne[:, 1])
fig.colorbar(im, ax=ax)

# ax=axes[1]
# im = ax.scatter(X_tsne[:, 1], X_tsne[:, 2], c=tsne_clusters)
# fig.colorbar(im, ax=ax)

# ax=axes[2]
# im = ax.scatter(X_tsne[:, 0], X_tsne[:, 2], c=tsne_clusters)
# fig.colorbar(im, ax=ax)
# im = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=radial_colours, cmap='viridis')
# fig.colorbar(im, ax=ax)

In [ ]:
# [np.float64(3.0726593301531495), np.float64(6.211561597848015), np.float64(9.876563283660191)]
new_df = stars_df
new_df['tsne_cluster'] = tsne_clusters
new_df['pca_cluster'] = pca_clusters
new_df['reg_cluster'] = reg_clusters

radial_bins = np.zeros(len(new_df), dtype='U20')
fx = np.zeros(len(new_df))
fy = np.zeros(len(new_df))

temp = []

radial_bins = np.zeros(len(new_df), dtype='U100')
for i, radius in enumerate(new_df['s1000r']):
    if radius > np.float64(6.211561597848015):
        r_sel = "outer"
        print(f"{radius:.02f} is inner")
        temp.append(0)
    elif radius > np.float64(3.0726593301531495):
        r_sel = "middle"
        print(f"{radius:.02f} is middle")
        temp.append(1)
    else:
        r_sel = "inner"
        print(f"{radius:.02f} is inner")


    radial_bins[i] = r_sel


new_df['radial_bin'] = radial_bins



In [ ]:



#aaa = ["yes" if pca_clusters[i] == 0 else "no" if pca_clusters[i] == 1 else "maybe" for i in range(len(pca_clusters))]
#morph_colours = ["blue" if morphs[j] == 1 else "purple" if morphs[j] == 0 else "green" for j in range(len(stars_df))]
morph_names = ["Arm" if j == 0 else "Bar" if j == 1 else "Disc" for j in np.array(morphs)]
radial_names = ["Inner" if j == "inner" else "Middle" if j == "middle" else "Outer" for j in new_df['radial_bin'].to_numpy()]

df = pd.DataFrame({
    'PCA': new_df['pca_cluster'],
    't-SNE': tsne_clusters,
    'reg' : reg_clusters,
    'Morphology': morph_names,
    'Radial Bin': radial_names,
})

#print(df['Radial'])
ncols=2
nrows=2
fig, axes = plt.subplots(nrows=nrows, ncols=ncols)
fig.set_size_inches(16,8)
axes = axes.ravel()

ax = axes[0]

pca_morph_count = df.groupby(['Morphology', 'PCA']).size().unstack(fill_value=0)
pca_morph_proportions = pca_morph_count.div(pca_morph_count.sum(axis=1), axis=0)
pca_radial_count = df.groupby(['Radial Bin', 'PCA']).size().unstack(fill_value=0)
pca_radial_proportions = pca_radial_count.div(pca_radial_count.sum(axis=1), axis=0)
tsne_morph_count = df.groupby(['Morphology', 'reg']).size().unstack(fill_value=0)
tsne_morph_proportions = tsne_morph_count.div(tsne_morph_count.sum(axis=1), axis=0)
tsne_radial_count = df.groupby(['Radial Bin', 'reg']).size().unstack(fill_value=0)

#print(tsne_radial_count)
tsne_radial_proportions = tsne_radial_count.div(tsne_radial_count.sum(axis=1), axis=0)


#pca_morph_count.plot(kind='bar', stacked=False, ax=axes[0])
pca_morph_proportions.plot(kind='bar', stacked=False, ax=axes[0], cmap='viridis', width=0.6)
pca_radial_proportions.plot(kind='bar', stacked=False, ax=axes[1], cmap='viridis', legend=False)
tsne_morph_proportions.plot(kind='bar', stacked=False, ax=axes[2], width=0.6)
tsne_radial_proportions.plot(kind='bar', stacked=False, ax=axes[3], width=0.6, legend=False)

axes[0].set_xlabel(None)
axes[0].set_ylabel("Proportion of Orbits")
axes[1].set_xlabel(None)
axes[2].set_ylabel("Proportion of Orbits")

axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)
axes[3].set_xticklabels(axes[3].get_xticklabels(), rotation=0)

fig.tight_layout()
#fig.savefig("figures/orbit_counts.pdf", format='pdf', bbox_inches='tight')


# ax.set(xlabel="category", ylabel='count')
# ax.tick_params(axis='x', rotation=0)
#ax.legend()


In [ ]:
c1df = new_df[new_df['pca_cluster'] == 4]['df'].sample(25)

nrows=5
ncols=5
fig, axes = plt.subplots(nrows=nrows, ncols=ncols)
fig.set_size_inches(4*ncols, 3*nrows)
axes = axes.ravel()

for i, df in enumerate(c1df):
    ax = axes[i]
    ax.plot(df['x'], df['y'])
    ax.scatter(df['x'], df['y'], c=df['snap'])